# Notebook for Siamese LSTM AUTOENCODER

## Imports

In [ ]:
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import LineString

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Bidirectional, Dense, TimeDistributed, Concatenate, RepeatVector, Dropout, LayerNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from  tensorflow.keras.optimizers import AdamW

from sklearn.metrics import mean_squared_error

from datetime import datetime

## Helper Functions

In [ ]:
# FUNCTION TO PREDICT UNSEEN SEQUENCES

def predict_reconstructed_sequences(network, sequence_a, sequence_b):
    combined_prediction = network.predict([sequence_a, sequence_b])
    half = combined_prediction.shape[1] // 2

    #slices the combined_prediction array to extract the first half columns
    reconstructed_sequence_a = combined_prediction[:, :half]
    
    #slices the combined_prediction array to extract the last half columns
    reconstructed_sequence_b = combined_prediction[:, half:]

    return reconstructed_sequence_a, reconstructed_sequence_b

## MODEL FUNCTIONS

In [ ]:
# CALCULATES THE PERPENDICULAR DISTANCE 
def perp_distance(args):

    point_a1, point_a2, point_b = args
    v = point_a2 - point_a1
    w = point_b  - point_a1

    # Squared length of the segment ‖v‖²  (add ε for numerical safety)
    vv = tf.reduce_sum(tf.square(v), axis=-1, keepdims=True) + tf.keras.backend.epsilon()

    # Projection scalar t = (w·v) / (‖v‖²)  — clip to [0,1] to stay on the *segment*
    t = tf.reduce_sum(w * v, axis=-1, keepdims=True) / vv
    t_clipped = tf.clip_by_value(t, 0.0, 1.0)

    # Nearest point on the segment to point_b
    nearest = point_a1 + t_clipped * v

    # Euclidean distance ‖point_b − nearest‖
    return tf.norm(point_b - nearest, axis=-1)


In [ ]:
# CALCULATES THE DISPLACEMENT LOSS

def displacement_loss(y_pred_b, y_true_a):
    # y_pred_b, y_true_a: [batch, seq_len, 2]
    a1 = y_true_a[:, :-1, :]  
    a2 = y_true_a[:, 1:, :] 
    v = a2 - a1

    # Expand for broadcasting
    p = tf.expand_dims(y_pred_b, 2) 
    a1 = tf.expand_dims(a1, 1)
    v  = tf.expand_dims(v, 1)

    w = p - a1
    vv = tf.reduce_sum(v**2, axis=-1, keepdims=True) + 1e-6
    t = tf.reduce_sum(w*v, axis=-1, keepdims=True) / vv
    t = tf.clip_by_value(t, 0.0, 1.0)

    nearest = a1 + t * v
    dists = tf.norm(p - nearest, axis=-1)
    min_dists = tf.reduce_min(dists, axis=-1)
    return tf.reduce_mean(min_dists)


In [ ]:
def combined_loss(y_true_a, y_pred_a, y_pred_b, y_true_b, alpha):
    y_true_a = tf.cast(y_true_a, tf.float32)
    y_true_b = tf.cast(y_true_b, tf.float32)
    y_pred_a = tf.cast(y_pred_a, tf.float32)
    y_pred_b = tf.cast(y_pred_b, tf.float32)

    mse_loss_a = tf.reduce_mean(tf.square(y_true_a - y_pred_a))
    mse_loss_b = tf.reduce_mean(tf.square(y_true_b - y_pred_b))

    #huber_loss_a = tf.keras.losses.huber(y_true_a, y_pred_a)
    #huber_loss_b = tf.keras.losses.huber(y_true_b, y_pred_b)

    disp_loss = displacement_loss(y_pred_b, y_true_a)

    # Optional debug
    tf.print(' MSE A:', mse_loss_a, ' MSE B:', mse_loss_b, ' Disp:', disp_loss)

    #return huber_loss_a + huber_loss_b - (alpha * disp_loss)
    return mse_loss_a + mse_loss_b - (alpha * disp_loss)

def siamese_loss_wrapper(alpha):
    def siamese_loss(y_true, y_pred):
        num_points = tf.shape(y_pred)[1] // 2

        y_true_a = y_true[:, :num_points, :]
        y_true_b = y_true[:, num_points:, :]
        y_pred_a = y_pred[:, :num_points, :]
        y_pred_b = y_pred[:, num_points:, :]

        return combined_loss(y_true_a, y_pred_a, y_pred_b, y_true_b, alpha)
    return siamese_loss

In [ ]:
# def create_autoencoder(input_shape, prefix=""):

#     inputs = Input(shape=input_shape)
#     #inputs = Input(shape=(None, 2)) #TBC padding and masking for different input lengths 

#     # Encoder
#     encoder_bi_lstm1 = Bidirectional(LSTM(128, return_sequences=True))(inputs)
#     encoder_bi_lstm1 = LayerNormalization()(encoder_bi_lstm1)
#     encoder_bi_lstm1 = Dropout(0.3)(encoder_bi_lstm1)

#     encoder_bi_lstm2 = Bidirectional(LSTM(64, return_sequences=True))(encoder_bi_lstm1)
#     encoder_bi_lstm2 = LayerNormalization()(encoder_bi_lstm2)
#     encoder_bi_lstm2 = Dropout(0.3)(encoder_bi_lstm2)

#     encoder_bi_lstm3 = Bidirectional(LSTM(32, return_sequences=True, return_state=True))
#     encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_bi_lstm3(encoder_bi_lstm2)
#     state_h = Concatenate()([forward_h, backward_h])
#     state_c = Concatenate()([forward_c, backward_c])
#     encoder_states = [state_h, state_c]

#     encoder_outputs = LayerNormalization()(encoder_outputs)
#     encoder_outputs = Dropout(0.3)(encoder_outputs)

#     # Decoder
#     forward_decoder_lstm = LSTM(32, return_sequences=True)
#     backward_decoder_lstm = LSTM(32, return_sequences=True, go_backwards=True)

#     forward_decoder_outputs = forward_decoder_lstm(encoder_outputs, initial_state=[forward_h, forward_c])
#     backward_decoder_outputs = backward_decoder_lstm(encoder_outputs, initial_state=[backward_h, backward_c])

#     decoder_outputs = Concatenate()([forward_decoder_outputs, backward_decoder_outputs])
#     decoder_outputs = LayerNormalization()(decoder_outputs)
#     decoder_outputs = Dropout(0.3)(decoder_outputs)

#     decoder_bi_lstm2 = Bidirectional(LSTM(64, return_sequences=True))(decoder_outputs)
#     decoder_bi_lstm2 = LayerNormalization()(decoder_bi_lstm2)
#     decoder_bi_lstm2 = Dropout(0.3)(decoder_bi_lstm2)

#     decoder_bi_lstm3 = Bidirectional(LSTM(128, return_sequences=True))(decoder_bi_lstm2)
#     decoder_bi_lstm3 = LayerNormalization()(decoder_bi_lstm3)
#     decoder_bi_lstm3 = Dropout(0.3)(decoder_bi_lstm3)

#     decoder_dense = TimeDistributed(Dense(2, activation='linear'))
#     decoder_outputs = decoder_dense(decoder_bi_lstm3)

#     autoencoder = Model(inputs, decoder_outputs)

#     return autoencoder

In [ ]:
# FUNCTION TO CREATE THE SIAMESE DATASET 
def make_siamese_dataset(a_noisy, b_noisy, a_clean, b_clean, shuffle=False, batch_size=32):
    inputs = (a_noisy, b_noisy)

    # combine clean targets into one tensor
    targets = np.stack([a_clean, b_clean], axis=1)  # shape: (N, 2, 64, 2) N = 1000

    dataset = tf.data.Dataset.from_tensor_slices((inputs, targets))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1024)

    return dataset.batch(batch_size)

## LOADING DATA

In [ ]:
PATH_TRAINING_A = '../data/preprocessing/normalized/normalized_local_original.npy'
PATH_TRAINING_B =  '../data/preprocessing/normalized/normalized_local_close.npy'

PATH_TRAINING_A_CLEAN = '../data/preprocessing/normalized/normalized_local_original.npy'
PATH_TRAINING_B_CLEAN = '../data/preprocessing/normalized/normalized_local_far.npy'


# Load full arrays
lines_a_noisy = np.load(PATH_TRAINING_A)[:20000]
lines_b_noisy = np.load(PATH_TRAINING_B)[:20000]
lines_a_clean = np.load(PATH_TRAINING_A_CLEAN)[:20000]
lines_b_clean = np.load(PATH_TRAINING_B_CLEAN)[:20000]

# Total number of samples
n_total = lines_a_noisy.shape[0]

# Compute split indices
n_train = int(0.7 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

# --- TRAIN ---
train_slice = slice(0, n_train)
lines_a_noisy_train = lines_a_noisy[train_slice]
lines_b_noisy_train = lines_b_noisy[train_slice]
lines_a_clean_train = lines_a_clean[train_slice]
lines_b_clean_train = lines_b_clean[train_slice]

# --- VALIDATION ---
val_slice = slice(n_train, n_train + n_val)
lines_a_noisy_val = lines_a_noisy[val_slice]
lines_b_noisy_val = lines_b_noisy[val_slice]
lines_a_clean_val = lines_a_clean[val_slice]
lines_b_clean_val = lines_b_clean[val_slice]

# --- TEST ---
test_slice = slice(n_train + n_val, n_total)
lines_a_noisy_test = lines_a_noisy[test_slice]
lines_b_noisy_test = lines_b_noisy[test_slice]
lines_a_clean_test = lines_a_clean[test_slice]
lines_b_clean_test = lines_b_clean[test_slice]

# Create training, validation, and test datasets
train_dataset = make_siamese_dataset(lines_a_noisy, lines_b_noisy, lines_a_clean, lines_b_clean, shuffle=False)
val_dataset   = make_siamese_dataset(lines_a_noisy_val, lines_b_noisy_val, lines_a_clean_val, lines_b_clean_val, shuffle=False)
test_dataset  = make_siamese_dataset(lines_a_clean_test, lines_b_clean_test, lines_a_noisy_test, lines_b_noisy_test, shuffle=False)

print(f"Train: {lines_a_noisy_train.shape}, Val: {lines_a_noisy_val.shape}, Test: {lines_a_noisy_test.shape}")

## CREATING MODEL

In [ ]:
# ORIGINAL
#Create two models, two input tensors and two reconstructed sequence tensors
# input_shape = (lines_a_noisy_train.shape[1], lines_a_noisy_train.shape[2])
# autoencoder_a = create_autoencoder(input_shape, 'A')
# autoencoder_b = create_autoencoder(input_shape, 'B')

# input_sequence_a = Input(shape=input_shape)
# input_sequence_b = Input(shape=input_shape)
# print(input_sequence_a)

# reconstructed_sequence_a = autoencoder_a(input_sequence_a)
# reconstructed_sequence_b = autoencoder_b(input_sequence_b)
# print(reconstructed_sequence_a)

# #Construct Model Architecture
# siamese_autoencoder = Model([input_sequence_a, input_sequence_b], Concatenate(axis=1)([reconstructed_sequence_a, reconstructed_sequence_b]))


# PRETRAINED AE 
pretrained_autoencoder = tf.keras.models.load_model(
    '../checkpoints/autoencoder/model/32_batches_50_epochs_huber_local_1511_1330.keras',
    compile=False
)

# for layer in pretrained_autoencoder.layers:
#     layer.trainable = False

input_shape = (lines_a_noisy_train.shape[1], lines_a_noisy_train.shape[2])
input_sequence_a = Input(shape=input_shape, name='line_a_input')
input_sequence_b = Input(shape=input_shape, name='line_b_input')

reconstructed_sequence_a = pretrained_autoencoder(input_sequence_a)
reconstructed_sequence_b = pretrained_autoencoder(input_sequence_b)

siamese_output = Concatenate(axis=1)([reconstructed_sequence_a, reconstructed_sequence_b])

siamese_autoencoder = Model(
    inputs=[input_sequence_a, input_sequence_b],
    outputs=siamese_output,
    name="siamese_autoencoder"
)

In [ ]:
#Define Training Parameters
loss = siamese_loss_wrapper(0.001)
epochs = 50
batch_size= 16
timestamp = datetime.now().strftime("%d%m_%H%M")

CHECKPOINT_CALLBACK = f'../checkpoints/siamese/weights/{batch_size}_batches_{epochs}_epochs_{timestamp}_pretrained.weights.h5'

#Checkpoint callback: saves models weights, when loss is smaller than before
checkpoint_callback = ModelCheckpoint(
    filepath= CHECKPOINT_CALLBACK,
    monitor='loss',
    mode="min",
    save_best_only=True,
    save_weights_only=True,
    verbose=1,
)

# Prevents model from overfitting due to early stopping 
early_stopping_callback = EarlyStopping(
    monitor='loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4) #TBC: AdamW

siamese_autoencoder.compile(optimizer=optimizer, loss=loss, metrics=['accuracy', 'mse', 'mae', 'cosine_similarity'])
lines_combined = np.concatenate([lines_a_clean_train, lines_b_clean_train], axis=1)
val_lines_combined = np.concatenate([lines_a_clean_val, lines_b_clean_val], axis=1)

# TRAINING 

In [ ]:
history = siamese_autoencoder.fit(
    [lines_a_noisy_train, lines_b_noisy_train], lines_combined,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(
      [lines_a_noisy_val, lines_b_noisy_val], val_lines_combined
    ),
    callbacks=[checkpoint_callback, early_stopping_callback])

siamese_autoencoder.load_weights(filepath=CHECKPOINT_CALLBACK)
siamese_autoencoder.save(f'../checkpoints/siamese/model/{batch_size}_batches_{epochs}_epochs_pretrained_local_{timestamp}.keras')
#np.save('../checkpoints/siamese/history/siamese_history.npy', history.history)

In [ ]:
siamese_autoencoder.summary()

In [ ]:
# plots the history
#history = np.load(''../checkpoints/siamese/history/siamese_history.npy')

def plot_history(history, epochs, batch_size, train_data_noisy):
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'], label='Training Loss', color='#143642')

    if 'val_loss' in history.history:
        plt.plot(history.history['val_loss'], label='Validation Loss', color='#EC9A29')

    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title(f'Training Loss Siamese LSTM AE {epochs} Epochs {batch_size} Batches and Training Data Shape: {train_data_noisy.shape}')
    plt.show()

plot_history(history, epochs, batch_size, lines_a_noisy)

## POST-TRAINING

In [ ]:
# Load model and weights 
siamese_autoencoder = tf.keras.models.load_model('../checkpoints/siamese/model/16_batches_50_epochs_pretrained_local_1811_1801.keras', custom_objects={'siamese_loss': loss})
#siamese_autoencoder = tf.keras.models.load_model(f'../checkpoints/siamese/model/{batch_size}_batches_{epochs}_epochs_rw_newLoss_newModel_{timestamp}.keras', custom_objects={'siamese_loss': loss})
#siamese_autoencoder.load_weights(filepath=f'../checkpoints/siamese/weights/{batch_size}_batches_{epochs}_epochs_{timestamp}.weights.h5')

In [ ]:
# PREDICTION 
pred_siamese_a, pred_siamese_b = predict_reconstructed_sequences(siamese_autoencoder, lines_a_noisy_test, lines_b_noisy_test)

In [ ]:
# PLOTTING RESULTS
plt.figure(figsize=(6, 10))
start_plot = 0
end_plot = start_plot + 1

# Plot training segments A
for i, segment in enumerate(lines_a_noisy_test[start_plot:end_plot,:,:]):
    plt.plot(segment[:, 0], segment[:, 1],
             color='#A8201A', linewidth=0.5,
             label='Original A' if i == 0 else "")

# Plot training segments B
for i, segment in enumerate(lines_b_noisy_test[start_plot:end_plot,:,:]):
    plt.plot(segment[:, 0], segment[:, 1],
             color='#143642', linewidth=0.5,
             label='Original B' if i == 0 else "")
    
# Plot training segments B
for i, segment in enumerate(lines_b_clean_test[start_plot:end_plot,:,:]):
    plt.plot(segment[:, 0], segment[:, 1],
             color="#1D431F", linewidth=0.5,
             label='Groundtruth B' if i == 0 else "")

# Plot predicted segments A
for i, segment in enumerate(pred_siamese_a[start_plot:end_plot,:,:]):
     plt.plot(segment[:, 0], segment[:, 1],
            color='#EC9A29', linestyle='--', linewidth=1,
            label='Prediction A' if i == 0 else "")

# Plot predicted segments B
for i, segment in enumerate(pred_siamese_b[start_plot:end_plot,:,:]):
     plt.plot(segment[:, 0], segment[:, 1],
            color="#286981", linestyle='--', linewidth=1,
            label='Prediction B' if i == 0 else "")

plt.title('Siamese LSTM AE Dataset')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Compute Euclidean error for all sequences and timesteps

all_errors_a = np.linalg.norm(lines_a_clean_test[:] - pred_siamese_a[:], axis=-1)
all_errors_b = np.linalg.norm(lines_b_clean_test[:] - pred_siamese_b[:], axis=-1)

# Flatten all errors into one 1D array
all_errors_flat_a = all_errors_a.flatten()  
all_errors_flat_b = all_errors_b.flatten()

# Plot all errors as one continuous line
plt.figure(figsize=(12, 4))
plt.plot(all_errors_flat_a, color='#EC9A29', linewidth=0.1, marker='o', linestyle='-', markersize=0.5, label='Errors A')
plt.plot(all_errors_flat_b, color='#143642', linewidth=0.5, marker='o', linestyle='-', markersize=0.5, label='Errors B')
plt.title('Euclidean Error for sequence B')
plt.xlabel('Sequence and Index (flattened)')
plt.ylabel('Euclidean Error')
plt.grid(True)
plt.show()

In [ ]:
# Compute MSE per point (squared error averaged over features)
mse_all = np.mean((lines_b_clean_test[:] - pred_siamese_b[:]) ** 2, axis=-1)  # shape: [num_sequences, sequence_length]

# Flatten for one continuous plot
mse_all_flat = mse_all.flatten()

plt.figure(figsize=(12, 4))
plt.plot(mse_all_flat, color='#EC9A29', linewidth=0.5, marker='o', markersize=0.5,linestyle='-')
plt.title('MSE for all Sequences and one Point')
plt.xlabel('Index (flattened)')
plt.ylabel('Mean Squared Error')
plt.grid(True)
plt.show()